In [ ]:
# ============================================================
# Grad-CAM vs Grad-CAM++  (best Fold)  +  Insertion / Deletion faithfulness

# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import gaussian_blur
from scipy.stats import wilcoxon
from tqdm.auto import tqdm

# -------------------- SETTINGS --------------------
FOLD        = 0
STEPS       = 50        # number of insertion/deletion steps (curve has STEPS+1 points)
TARGET      = "pred"    # "pred" = explain predicted class | "true" = explain ground-truth class
MAX_IMAGES  = None      # e.g. 100 for a quick test, None = whole fold-0 test set
BLUR_K, BLUR_S = 51, 25.0   # blur baseline used for insertion
n_cls = len(CLASSES)
_trapz = getattr(np, "trapezoid", None) or np.trapz

# -------------------- LOAD FOLD-0 MODEL --------------------
model0 = build_model(pretrained=False)
ckpt = os.path.join(OUT_DIR, f"best_proposed_fold{FOLD}.pth")
model0.load_state_dict(torch.load(ckpt, map_location=device))
model0.eval()
print(f"Loaded {ckpt}")

# -------------------- CAM EXTRACTOR --------------------
# Target layer = BiFPN fused feature map (highest-res of the last 3 scales).
# If BiFPN is off, falls back to the last attention block output.
class CamExtractor:
    def __init__(self, model):
        layer = model.fusion if model.use_bifpn else model.attns[-1]
        self.act = None
        self.h = layer.register_forward_hook(lambda m, i, o: setattr(self, "act", o))
    def remove(self):
        self.h.remove()

ext = CamExtractor(model0)

def _norm01(cam):
    mn = cam.flatten(1).min(1)[0].view(-1, 1, 1)
    mx = cam.flatten(1).max(1)[0].view(-1, 1, 1)
    return (cam - mn) / (mx - mn + 1e-8)

def compute_cams(model, x, target=None):
    """Returns Grad-CAM, Grad-CAM++ maps (B,H,W in [0,1]), target idx, pred idx."""
    x = x.to(device)
    with torch.enable_grad():
        logits = model(x)
        pred = logits.argmax(1)
        tgt = pred if target is None else target.to(device)
        score = logits.gather(1, tgt[:, None]).sum()
        A = ext.act
        g = torch.autograd.grad(score, A)[0]
    A = A.detach(); g = g.detach()

    # Grad-CAM
    w = g.mean(dim=(2, 3), keepdim=True)
    cam_g = F.relu((w * A).sum(1))

    # Grad-CAM++
    g2, g3 = g ** 2, g ** 3
    sumA = A.sum(dim=(2, 3), keepdim=True)
    alpha = g2 / (2 * g2 + sumA * g3 + 1e-8)
    alpha = torch.where(g != 0, alpha, torch.zeros_like(alpha))
    w_pp = (alpha * F.relu(g)).sum(dim=(2, 3), keepdim=True)
    cam_pp = F.relu((w_pp * A).sum(1))

    size = x.shape[-2:]
    up = lambda c: _norm01(F.interpolate(c[:, None], size=size, mode="bilinear",
                                         align_corners=False)[:, 0])
    return up(cam_g), up(cam_pp), tgt, pred

# -------------------- INSERTION / DELETION --------------------
@torch.no_grad()
def ins_del_curves(model, x, sal, tgt, steps=STEPS):
    """x:(3,H,W), sal:(H,W). Returns insertion & deletion prob curves (steps+1,)
    of the softmax probability of class `tgt`."""
    C, H, W = x.shape
    HW = H * W
    order = torch.argsort(sal.flatten(), descending=True)
    rank = torch.empty_like(order)
    rank[order] = torch.arange(HW, device=x.device)
    ks = torch.linspace(0, HW, steps + 1, device=x.device).long()
    masks = (rank[None, :] < ks[:, None]).view(steps + 1, 1, H, W)   # top-k pixels

    xb = x[None]
    blur = gaussian_blur(xb, kernel_size=[BLUR_K, BLUR_K], sigma=[BLUR_S, BLUR_S])
    ins_in = torch.where(masks, xb, blur)                    # start blurred -> reveal
    del_in = torch.where(masks, torch.zeros_like(xb), xb)    # start original -> remove

    p_ins = model(ins_in).float().softmax(1)[:, tgt].cpu().numpy()
    p_del = model(del_in).float().softmax(1)[:, tgt].cpu().numpy()
    return p_ins, p_del

# -------------------- MAIN LOOP --------------------
te_loader = make_loader(folds[FOLD]["test"], eval_tf, False)
xs = np.linspace(0, 1, STEPS + 1)

records = []
curves = {k: [] for k in ["ins_gc", "del_gc", "ins_gpp", "del_gpp"]}
demo = {}          # one correctly classified example per class (qualitative figure)
n_done = 0

for xb, yb in tqdm(te_loader, desc=f"Fold {FOLD}: CAM + Ins/Del"):
    xb = xb.to(device)
    cam_g, cam_pp, tgt, pred = compute_cams(
        model0, xb, target=None if TARGET == "pred" else yb)

    for i in range(xb.shape[0]):
        if MAX_IMAGES is not None and n_done >= MAX_IMAGES:
            break
        t = int(tgt[i]); x_i = xb[i]
        ins_g, del_g = ins_del_curves(model0, x_i, cam_g[i], t)
        ins_p, del_p = ins_del_curves(model0, x_i, cam_pp[i], t)
        curves["ins_gc"].append(ins_g);  curves["del_gc"].append(del_g)
        curves["ins_gpp"].append(ins_p); curves["del_gpp"].append(del_p)
        records.append({
            "target": t, "true": int(yb[i]), "pred": int(pred[i]),
            "ins_gradcam": _trapz(ins_g, xs), "del_gradcam": _trapz(del_g, xs),
            "ins_gradcampp": _trapz(ins_p, xs), "del_gradcampp": _trapz(del_p, xs),
        })
        if int(pred[i]) == int(yb[i]) and int(yb[i]) not in demo:
            demo[int(yb[i])] = (x_i.cpu(), cam_g[i].cpu(), cam_pp[i].cpu())
        n_done += 1
    if MAX_IMAGES is not None and n_done >= MAX_IMAGES:
        break

ext.remove()
df = pd.DataFrame(records)
curves = {k: np.array(v) for k, v in curves.items()}
df.to_csv(os.path.join(OUT_DIR, f"per_image_faithfulness_fold{FOLD}.csv"), index=False)
np.savez(os.path.join(OUT_DIR, f"insdel_curves_fold{FOLD}.npz"), **curves)

# -------------------- PER-CLASS TABLE --------------------
def ms(v):
    return f"{np.mean(v):.4f} ± {np.std(v):.4f}"

rows = []
groups = [(CLASSES[c], df[df.target == c]) for c in range(n_cls)] + [("Overall", df)]
for name, d in groups:
    if len(d) == 0:
        continue
    rows.append({
        "Class": name, "N": len(d),
        "Ins Grad-CAM ↑":   ms(d.ins_gradcam),
        "Ins Grad-CAM++ ↑": ms(d.ins_gradcampp),
        "Del Grad-CAM ↓":   ms(d.del_gradcam),
        "Del Grad-CAM++ ↓": ms(d.del_gradcampp),
        "Ins-Del Grad-CAM ↑":   f"{np.mean(d.ins_gradcam - d.del_gradcam):.4f}",
        "Ins-Del Grad-CAM++ ↑": f"{np.mean(d.ins_gradcampp - d.del_gradcampp):.4f}",
    })
table = pd.DataFrame(rows)
print(f"\n===== Fold {FOLD}: Insertion / Deletion AUC per class (mean ± std, target={TARGET}) =====")
print(table.to_string(index=False))
table.to_csv(os.path.join(OUT_DIR, f"insdel_table_fold{FOLD}.csv"), index=False)

# paired significance (overall)
if len(df) > 5:
    p_ins = wilcoxon(df.ins_gradcampp, df.ins_gradcam).pvalue
    p_del = wilcoxon(df.del_gradcampp, df.del_gradcam).pvalue
    print(f"\nPaired Wilcoxon (Grad-CAM++ vs Grad-CAM): insertion p = {p_ins:.4g}, deletion p = {p_del:.4g}")

# -------------------- FIGURE: MEAN INSERTION (left) / DELETION (right) --------------------
def plot_curve(ax, key_gc, key_pp, title, ylabel):
    for key, lab, col in [(key_gc, "Grad-CAM", "tab:blue"), (key_pp, "Grad-CAM++", "tab:red")]:
        c = curves[key]
        m = c.mean(0)
        ci = 1.96 * c.std(0) / np.sqrt(len(c))
        auc_v = _trapz(m, xs)
        ax.plot(xs * 100, m, color=col, lw=2.2, label=f"{lab} (AUC = {auc_v:.3f})")
        ax.fill_between(xs * 100, m - ci, m + ci, color=col, alpha=0.2)
    ax.set_title(title); ax.set_xlabel(ylabel); ax.set_ylabel("Predicted-class probability")
    ax.set_xlim(0, 100); ax.set_ylim(0, 1.02)
    ax.grid(True, alpha=0.3); ax.legend(loc="best")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_curve(axes[0], "ins_gc", "ins_gpp", "Insertion (higher is better)", "Pixels inserted (%)")
plot_curve(axes[1], "del_gc", "del_gpp", "Deletion (lower is better)", "Pixels deleted (%)")
plt.suptitle(f"Fold {FOLD}: Mean Insertion / Deletion Curves (Grad-CAM vs Grad-CAM++)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"insdel_curves_fold{FOLD}.png"), dpi=300, bbox_inches="tight")
plt.show()

# -------------------- QUALITATIVE FIGURE --------------------
cls_ids = sorted(demo.keys())
if cls_ids:
    fig, axes = plt.subplots(len(cls_ids), 3, figsize=(9, 3 * len(cls_ids)))
    axes = np.atleast_2d(axes)
    for r, c in enumerate(cls_ids):
        img, cg, cp = demo[c]
        img = img.permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)   # display-only rescale
        axes[r, 0].imshow(img); axes[r, 0].set_ylabel(CLASSES[c], fontsize=11)
        axes[r, 1].imshow(img); axes[r, 1].imshow(cg.numpy(), cmap="jet", alpha=0.5)
        axes[r, 2].imshow(img); axes[r, 2].imshow(cp.numpy(), cmap="jet", alpha=0.5)
        if r == 0:
            axes[r, 0].set_title("Input"); axes[r, 1].set_title("Grad-CAM"); axes[r, 2].set_title("Grad-CAM++")
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"gradcam_vs_gradcampp_fold{FOLD}.png"), dpi=300, bbox_inches="tight")
    plt.show()

print(f"\n✅ Saved in {OUT_DIR}:")
print(f"   insdel_table_fold{FOLD}.csv, per_image_faithfulness_fold{FOLD}.csv, insdel_curves_fold{FOLD}.npz")
print(f"   insdel_curves_fold{FOLD}.png, gradcam_vs_gradcampp_fold{FOLD}.png")